# AI Image FFT Detector - Colab runner

Run setup, then only the authorized phase. Phase 1 collects original files; Phase 2 audits and freezes splits. No classifier training yet. CPU only; all code is on main. Save actual outputs for team review.

In [5]:
from google.colab import drive
drive.mount("/content/drive")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Clone the project
Clone main from the configured repository. An existing checkout is not overwritten; use a fresh session when you need a new checkout.

In [10]:
from pathlib import Path
import os
import subprocess

REPO_URL = "https://github.com/Rahul2-5/SpectraCheck.git"
PROJECT_DIR = Path("/content/ai-image-fft-detector")
if REPO_URL == "REPLACE_WITH_YOUR_GITHUB_REPOSITORY_URL":
    raise ValueError("Set REPO_URL to your GitHub repository URL first")

if PROJECT_DIR.exists():
    print(f"Directory {PROJECT_DIR} already exists. Skipping clone and pulling latest changes...")
    os.chdir(PROJECT_DIR)
    subprocess.run(["git", "pull"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", "main", REPO_URL, str(PROJECT_DIR)], check=True)
    os.chdir(PROJECT_DIR)

print("Working directory:", Path.cwd())

Directory /content/ai-image-fft-detector already exists. Skipping clone and pulling latest changes...
Working directory: /content/ai-image-fft-detector


In [7]:
%pip install -r /content/ai-image-fft-detector/requirements.txt
%pip install --no-deps -e /content/ai-image-fft-detector


Obtaining file:///content/ai-image-fft-detector
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for ai-image-fft-detector (pyproject.toml) ... done
  Created wheel for ai-image-fft-detector: filename=ai_image_fft_detector-0.1.0-0.editable-py3-none-any.whl size=9657 sha256=0c5f4a3e1a336e4bff72a270ed2b95f1cb46fad153ac749ac572b0bebeccb202
  Stored in directory: /tmp/pip-ephem-wheel-cache-ej3nvhjb/wheels/c9/05/2f/d6e4755b58ddf89231e97da982a277fc7911457522463e1f16
Successfully built ai-image-fft-detector
  Attempting uninstall: ai-image-fft-detector
    Found existing installation: ai-image-fft-detector 0.1.0
    Uninstalling ai-image-fft-detector-0.1.0:
      Successfully uninstalled ai-image-fft-detector-0.1.0


If Colab asks you to restart after installation, use Runtime > Restart session, then continue below (do not rerun the clone cell). This avoids mixing preloaded library versions with newly installed ones.

In [11]:
from pathlib import Path
import importlib
import importlib.metadata as metadata
import os
import platform
import subprocess
import sys

PROJECT_DIR = Path("/content/ai-image-fft-detector")
os.chdir(PROJECT_DIR)
print("Python:", platform.python_version())
print("Branch:", subprocess.check_output(["git", "branch", "--show-current"], text=True).strip())
print("Commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())

# Run pip check but capture any error peacefully instead of failing
result = subprocess.run([sys.executable, "-m", "pip", "check"], capture_output=True, text=True)
if result.returncode != 0:
    print("\n[WARNING] pip check returned non-zero code. Dependency conflicts exist:")
    print(result.stdout or result.stderr)
else:
    print("pip check: OK")

modules = {"pillow": "PIL", "scikit-learn": "sklearn", "pyyaml": "yaml", "imagehash": "imagehash", "huggingface-hub": "huggingface_hub"}
for line in (PROJECT_DIR / "requirements.txt").read_text().splitlines():
    if not line.strip() or line.startswith("#"):
        continue
    package, expected = line.split("==")
    installed = metadata.version(package)
    assert installed == expected, f"Version mismatch: {package}: {installed} != {expected}"
    importlib.import_module(modules.get(package, package))
    print(f"{package}=={installed}")

from aidetect.io_utils import config_hash, load_config
cfg = load_config("config.yaml")
print("Config SHA-256:", config_hash(cfg))

Python: 3.13.16
Branch: main
Commit: bbb0733c1d58bb39ee1973bd4509a577ab3bfc30

[WARNING] pip check returned non-zero code. Dependency conflicts exist:
ipython 7.34.0 requires jedi, which is not installed.
google-colab 1.0.0 has requirement pandas==2.2.3, but you have pandas 3.1.0.
gradio 6.30.0 has requirement pillow<13.0,>=8.0, but you have pillow 13.0.0.
cudf-cu13 26.6.0 has requirement pandas<2.4.0,>=2.0, but you have pandas 3.1.0.
cudf-cu13 26.6.0 has requirement pyarrow<24,>=19.0.0, but you have pyarrow 26.0.0.
opentelemetry-semantic-conventions 0.63b1 has requirement opentelemetry-api==1.42.1, but you have opentelemetry-api 1.45.1.
google-adk 2.7.1 has requirement opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.1.
numba 0.61.2 has requirement numpy<2.3,>=1.24, but you have numpy 2.4.6.
opentelemetry-sdk 1.42.1 has requirement opentelemetry-api==1.42.1, but you have opentelemetry-api 1.45.1.
dask-cudf-cu13 26.6.0 has requirement pandas<2.4.0,>=2.0, but you ha

/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


pandas==3.1.0
pyarrow==26.0.0
pillow==13.0.0
scikit-learn==1.9.1
xgboost==3.2.0
statsmodels==0.15.0
imagehash==4.3.2
matplotlib==3.11.2
datasets==5.1.0
huggingface-hub==2.3.0
pyyaml==6.0.3
joblib==1.6.0
pytest==9.1.1
fastapi==0.162.0
uvicorn==0.54.0
httpx==0.28.1
Config SHA-256: ce79d3ef5f012768bbc377db9cdd9226377a1449a13a7725d9d49189f3549dce


In [12]:
subprocess.run([sys.executable, "-m", "pytest", "-q"], cwd=PROJECT_DIR, check=True)


CompletedProcess(args=['/usr/bin/python3', '-m', 'pytest', '-q'], returncode=0)

## Phase 0 review
Record setup results and check GitHub Actions. If Phase 1 has been authorized, continue below; otherwise stop here.

## Phase 1 - Original image collection
Download CIFAKE manually from [Kaggle](https://www.kaggle.com/datasets/birdy654/cifake-real-and-ai-generated-synthetic-images) and extract it on Drive. Set the folder below to the parent of train/REAL, train/FAKE, test/REAL, test/FAKE. The pilot is separate from main data. Keep this session on CPU.

In [13]:
CIFAKE_SOURCE_DIR = Path("/content/drive/MyDrive/cifake")
if not CIFAKE_SOURCE_DIR.is_dir():
    raise FileNotFoundError("Set CIFAKE_SOURCE_DIR to your extracted CIFAKE folder on Drive")
subprocess.run([sys.executable, "scripts/download_pilot.py", "--source-dir", str(CIFAKE_SOURCE_DIR)], cwd=PROJECT_DIR, check=True)


FileNotFoundError: Set CIFAKE_SOURCE_DIR to your extracted CIFAKE folder on Drive

### Main collection
The next command scans Tiny-GenImage metadata, counts every generator, then saves only SD15 images and a random sample of real images. It may transfer the full upstream Parquet dataset; a subset does not guarantee a small download. Rerun after a disconnection to reuse unchanged saved files. Exit code 2 means the SD15 count is too small: outputs are saved, but consult the supervisor before Phase 2.

In [ ]:
subprocess.run([sys.executable, "scripts/download_main.py"], cwd=PROJECT_DIR, check=True)


In [ ]:
import json
import pandas as pd
from PIL import Image as PILImage
from IPython.display import display

cfg = load_config(PROJECT_DIR / "config.yaml")
for collection in ("pilot", "main"):
    root = Path(cfg["paths"]["data_root"]) / collection
    manifest = pd.read_csv(root / "manifest.csv")
    record = json.loads((root / "acquisition.json").read_text())
    assert record["status"] == "complete", "Acquisition did not finish"
    listed = set(manifest["rel_path"])
    actual = {p.relative_to(root).as_posix() for folder in ("real", "ai") for p in (root / folder).glob("*") if p.is_file() and p.suffix != ".tmp"}
    assert len(manifest) == len(listed) and listed == actual, "Manifest/file counts differ"
    assert all((root / row.rel_path).stat().st_size == row.n_bytes for row in manifest.itertuples())
    print(collection, "saved counts:", manifest.groupby("label").size().to_dict())
    print(collection, "source generator counts:", record["generator_counts"])
    if collection == "main":
        assert record["saved_counts"]["ai"] >= cfg["dataset"]["minimum_ai_images"], "STOP: ask supervisor about insufficient SD15 data"
        print("Record SD15 and SD14 counts in the team tracker.")
    for label in (0, 1):
        samples = manifest[manifest["label"] == label].sample(n=5, random_state=cfg["seed"])
        for relative in samples["rel_path"]:
            with PILImage.open(root / relative) as image:
                image.load()
                print(collection, "label", label, relative, image.size, image.format)
                display(image.copy())


## Phase 1 review
Confirm generator counts, supervisor decision if SD15 is below the configured minimum, matching manifest/file counts, and five readable images per class. Continue only if Phase 2 is authorized. Keep real-data checks pending when their outputs are not available.

## Phase 2 - Audit before building the detector
Use the complete main collection already on Drive. Skip Phase 1 commands if it is already downloaded. The audit measures file format, quality and size, excludes undersized images before freezing, balances classes, and tests metadata-only shortcuts. It preserves originals. If a probe still exceeds 0.60 after the preplanned fix, STOP and review the report; do not weaken the threshold.

In [ ]:
subprocess.run([sys.executable, "scripts/run_audit.py"], cwd=PROJECT_DIR, check=True)


In [ ]:
from IPython.display import Markdown, Image as DisplayImage, display

cfg = load_config(PROJECT_DIR / "config.yaml")
REPORTS_DIR = Path(cfg["paths"]["reports_root"])
report_text = (REPORTS_DIR / "audit_report.md").read_text()
display(Markdown(chr(10).join(line for line in report_text.splitlines() if not line.startswith("!["))))
for figure in sorted((REPORTS_DIR / "figures").glob("audit_*.png")):
    display(DisplayImage(filename=str(figure)))


In [ ]:
subprocess.run([sys.executable, "scripts/make_splits.py"], cwd=PROJECT_DIR, check=True)


In [ ]:
import json
import pandas as pd
from aidetect.io_utils import sha256_of_file

DATA_ROOT = Path(cfg["paths"]["data_root"])
ARTIFACTS_DIR = Path(cfg["paths"]["artifacts_root"])
audit_result = json.loads((DATA_ROOT / "audit/audit_result.json").read_text())
assert audit_result["status"] == "passed"
assert audit_result["final_probe"]["max_accuracy"] <= cfg["audit"]["shortcut_max_accuracy"]
splits = pd.read_csv(DATA_ROOT / "splits.csv", dtype={"id": str, "group_id": str})
counts = splits.groupby(["split", "label"]).size().unstack(fill_value=0)
assert (counts[0] == counts[1]).all()
assert (splits.groupby("group_id")["split"].nunique() == 1).all()
for name, target in cfg["split"].items():
    fraction = (splits["split"] == name).mean()
    assert abs(fraction - target) <= cfg["splitting"]["proportion_tolerance"] + 1e-9
digest = sha256_of_file(DATA_ROOT / "splits.csv")
assert digest == (ARTIFACTS_DIR / "splits.sha256").read_text().strip()
print("Final metadata probes:", audit_result["final_probe"])
print("Class counts (real=0, AI=1):")
print(counts)
print("Groups crossing splits: 0")
print("Exclusions:", audit_result["exclusion_reasons"])
print("Splits SHA-256:", digest)


In [ ]:
import shutil

BACKUP_DIR = DATA_ROOT / "audit/review_backup"
BACKUP_DIR.mkdir(parents=True, exist_ok=True)
for filename in ("splits.sha256", "splits.json"):
    shutil.copy2(ARTIFACTS_DIR / filename, BACKUP_DIR / filename)
for source in REPORTS_DIR.glob("audit_*"):
    if source.is_file():
        shutil.copy2(source, BACKUP_DIR / source.name)
(BACKUP_DIR / "figures").mkdir(exist_ok=True)
for source in (REPORTS_DIR / "figures").glob("audit_*.png"):
    shutil.copy2(source, BACKUP_DIR / "figures" / source.name)
print("Review records backed up to Drive:", BACKUP_DIR)
print("After team review, commit actual artifacts/splits.sha256 and artifacts/splits.json to main.")


## STOP - Phase 2 team review
Review format/size/quality tables, both before/after probe scores, exclusions, exact class balance, and zero crossing duplicate groups. Record and commit the actual split hash/provenance. If blocked or infeasible, share the exact output; do not start Phase 3 or silently change the data. No frequency features have been computed.

## Phase 3 - Shared preprocessing
Run this after the reviewed Phase 2 split is present. The check uses only 20 deterministic training rows per class, displays successful preprocessed arrays, and counts every `smaller than target` rejection. It does not inspect validation/test rows or create feature files.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from aidetect.io_utils import load_config
from aidetect.preprocess import preprocess_image

cfg = load_config(PROJECT_DIR / "config.yaml")
DATA_ROOT = Path(cfg["paths"]["data_root"])
active = pd.read_csv(DATA_ROOT / "audit/active_manifest.csv", dtype={"id": str})
frozen = pd.read_csv(DATA_ROOT / "splits.csv", dtype={"id": str})
train = active.merge(frozen[["id", "split"]], on="id", how="inner", validate="one_to_one")
train = train[train["split"] == "train"].sort_values(["label", "id"])
previews = {0: [], 1: []}
smaller_than_target = {0: 0, 1: 0}
for label in (0, 1):
    selected = train[train["label"] == label].head(20)
    if len(selected) != 20:
        raise ValueError(f"Need 20 training images for label {label}; found {len(selected)}")
    for row in selected.itertuples(index=False):
        try:
            array = preprocess_image(DATA_ROOT / row.rel_path, cfg)
        except ValueError as exc:
            if "smaller than target" not in str(exc):
                raise
            smaller_than_target[label] += 1
        else:
            previews[label].append((row.id, array))

for label, title in ((0, "real = 0"), (1, "AI-generated = 1")):
    figure, axes = plt.subplots(4, 5, figsize=(10, 10), squeeze=False)
    for axis in axes.ravel():
        axis.axis("off")
    for axis, (identifier, array) in zip(axes.ravel(), previews[label]):
        axis.imshow(array, cmap="gray", vmin=0, vmax=1)
        axis.set_title(identifier, fontsize=7)
        axis.axis("off")
    figure.suptitle(title)
    plt.tight_layout()
    plt.show()

print("smaller than target counts:", smaller_than_target)

## Phase 4 - Frequency features
Run this after the reviewed Phase 2 split and Phase 3 preprocessing check. It uses 200 deterministic training rows per class only, plots mean log spectra and mean RAPSD with 95% bands, and saves figures under `reports/figures/`. It never reads validation/test rows.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from aidetect.features import power_spectrum, rapsd
from aidetect.io_utils import load_config
from aidetect.preprocess import preprocess_image

cfg = load_config(PROJECT_DIR / "config.yaml")
DATA_ROOT = Path(cfg["paths"]["data_root"])
REPORTS_DIR = PROJECT_DIR / cfg["paths"]["reports_root"]
FIGURES_DIR = REPORTS_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
active = pd.read_csv(DATA_ROOT / "audit/active_manifest.csv", dtype={"id": str})
frozen = pd.read_csv(DATA_ROOT / "splits.csv", dtype={"id": str})
train = active.merge(frozen[["id", "split"]], on="id", how="inner", validate="one_to_one")
train = train[train["split"] == "train"].sort_values(["label", "id"])
frequency = cfg.get("frequency", {})
epsilon = float(frequency.get("epsilon", 1.0e-12))
rings = int(cfg["rapsd_rings"])
spectra = {}
rapsd_values = {}
for label in (0, 1):
    selected = train[train["label"] == label].head(200)
    if len(selected) != 200:
        raise ValueError(f"Need 200 training images for label {label}; found {len(selected)}")
    class_spectra = []
    class_rapsd = []
    for row in selected.itertuples(index=False):
        image = preprocess_image(DATA_ROOT / row.rel_path, cfg)
        power = power_spectrum(image, window=frequency.get("hann_window", True))
        class_spectra.append(np.log10(np.maximum(power, epsilon)))
        class_rapsd.append(rapsd(power, rings=rings, epsilon=epsilon))
    spectra[label] = np.stack(class_spectra)
    rapsd_values[label] = np.stack(class_rapsd)

mean_spectra = {label: values.mean(axis=0) for label, values in spectra.items()}
vmin = min(values.min() for values in mean_spectra.values())
vmax = max(values.max() for values in mean_spectra.values())
figure, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)
for axis, label in zip(axes, (0, 1)):
    image = axis.imshow(mean_spectra[label], cmap="magma", vmin=vmin, vmax=vmax)
    axis.set_title("real = 0" if label == 0 else "AI-generated = 1")
    axis.set_xlabel("shifted FFT column")
    axis.set_ylabel("shifted FFT row")
figure.colorbar(image, ax=axes, label="mean log10 power")
figure.savefig(FIGURES_DIR / "phase_4_mean_log_spectrum.png", dpi=cfg["audit"]["figure_dpi"])
plt.show()

figure, axis = plt.subplots(figsize=(9, 5))
x_axis = np.arange(1, rings + 1)
for label, color, title in ((0, "tab:blue", "real = 0"), (1, "tab:orange", "AI-generated = 1")):
    values = rapsd_values[label]
    mean = values.mean(axis=0)
    band = 1.96 * values.std(axis=0, ddof=1) / np.sqrt(values.shape[0])
    axis.plot(x_axis, mean, color=color, label=title)
    axis.fill_between(x_axis, mean - band, mean + band, color=color, alpha=0.2)
axis.set_xlabel("RAPSD ring")
axis.set_ylabel("log10 mean power")
axis.set_title("Training-set mean RAPSD with 95% bands")
axis.legend()
figure.savefig(FIGURES_DIR / "phase_4_mean_rapsd.png", dpi=cfg["audit"]["figure_dpi"])
plt.show()
print("Saved:", FIGURES_DIR / "phase_4_mean_log_spectrum.png")
print("Saved:", FIGURES_DIR / "phase_4_mean_rapsd.png")

## Phase 5 - Pixel baseline and feature extraction
This phase extracts both feature sets in resumable batches of the configured size. Train and validation batches may be read for checks; test features are written to `features_root/test_sealed/` but are never read here. The official reader refuses test rows until `artifacts/FREEZE.lock` exists in Phase 7.

In [ ]:
import json
import numpy as np
import pandas as pd

cfg = load_config(PROJECT_DIR / "config.yaml")
DATA_ROOT = Path(cfg["paths"]["data_root"])
ARTIFACTS_ROOT = Path(cfg["paths"]["artifacts_root"])
FEATURES_ROOT = Path(cfg["paths"]["features_root"])
FREEZE_LOCK = ARTIFACTS_ROOT / "FREEZE.lock"
if FREEZE_LOCK.exists():
    raise RuntimeError("Phase 5 must run before the Phase 7 freeze lock exists")
active = pd.read_csv(DATA_ROOT / "audit/active_manifest.csv", dtype={"id": str})
frozen = pd.read_csv(DATA_ROOT / "splits.csv", dtype={"id": str, "group_id": str})
joined = frozen.merge(active[["id", "rel_path"]], on="id", how="inner", validate="one_to_one")
assert len(joined) == len(frozen), "Active manifest and frozen split IDs differ"
print("Frozen row counts:", frozen.groupby(["split", "label"]).size().to_dict())
print("Features root:", FEATURES_ROOT)
print("Configured batch size:", cfg["batch_size"])


In [ ]:
for split in ("train", "val"):
    print(f"Extracting {split}...")
    subprocess.run([sys.executable, "scripts/extract_features.py", "--split", split], cwd=PROJECT_DIR, check=True)


In [ ]:
print("Writing test features to the sealed folder; this command does not read feature output.")
subprocess.run([sys.executable, "scripts/extract_features.py", "--split", "test"], cwd=PROJECT_DIR, check=True)


In [ ]:
from aidetect.feature_extraction import read_feature_batches

for split in ("train", "val"):
    frame = read_feature_batches(FEATURES_ROOT, split, artifacts_root=ARTIFACTS_ROOT)
    expected = int((frozen["split"] == split).sum())
    assert len(frame) == expected, f"{split} row count mismatch"
    numeric = frame.select_dtypes(include=[np.number])
    assert np.isfinite(numeric.to_numpy()).all(), f"{split} contains NaN/Inf"
    print(split, "rows:", len(frame), "finite numeric values: yes")
try:
    read_feature_batches(FEATURES_ROOT, "test", artifacts_root=ARTIFACTS_ROOT)
except PermissionError as error:
    print("Expected sealed-test denial:", error)
else:
    raise AssertionError("test_sealed was readable before FREEZE.lock")
test_batches = sorted((FEATURES_ROOT / "test_sealed").glob("batch_*.parquet"))
assert test_batches, "No test feature batches were written"
print("Test feature batches written:", len(test_batches), "(values remain sealed)")


## Phase 6 - Train and select models
Run only after the team has reviewed the Phase 2 immutable split and the Phase 5 train/validation feature checks. This phase reads train and validation features only. It does not access `test_sealed`; do not create the Phase 7 freeze lock yet.
The script selects frequency settings on train/validation, tunes the six matched pipelines on train folds, reports validation metrics, and records the validation-selected frequency model. Feature importance is descriptive only.


In [ ]:
from aidetect.io_utils import sha256_of_file

cfg = load_config(PROJECT_DIR / "config.yaml")
DATA_ROOT = Path(cfg["paths"]["data_root"])
ARTIFACTS_ROOT = Path(cfg["paths"]["artifacts_root"])
FEATURES_ROOT = Path(cfg["paths"]["features_root"])
if (ARTIFACTS_ROOT / "FREEZE.lock").exists():
    raise RuntimeError("Phase 6 must finish before the Phase 7 freeze")
split_path = DATA_ROOT / "splits.csv"
split_digest = (ARTIFACTS_ROOT / "splits.sha256").read_text().strip()
assert sha256_of_file(split_path) == split_digest, "Frozen split hash mismatch"
for split in ("train", "val"):
    batches = sorted((FEATURES_ROOT / split).glob("batch_*.parquet"))
    assert batches, f"Missing {split} feature batches"
print("Frozen split SHA-256:", split_digest)
print("Train/validation batch counts:", {split: len(list((FEATURES_ROOT / split).glob("batch_*.parquet"))) for split in ("train", "val")})


In [ ]:
subprocess.run([sys.executable, "scripts/train_models.py"], cwd=PROJECT_DIR, check=True)

# Persist ignored model binaries immediately so a Colab runtime restart does not lose Phase 6 outputs.
import json
import shutil
from aidetect.io_utils import sha256_of_file
cfg = load_config(PROJECT_DIR / "config.yaml")
ARTIFACTS_ROOT = Path(cfg["paths"]["artifacts_root"])
DRIVE_ROOT = Path("/content/drive/MyDrive/aidetect")
MODEL_BACKUP_ROOT = DRIVE_ROOT / "phase_6_models"
MODEL_BACKUP_MANIFEST = DRIVE_ROOT / "phase_6_models.sha256.json"
model_names = [f"{feature_set}_{model}.joblib" for feature_set in ("frequency", "pixel") for model in ("svm", "rf", "xgb")]
if MODEL_BACKUP_MANIFEST.is_file():
    manifest = json.loads(MODEL_BACKUP_MANIFEST.read_text())
    assert set(manifest["models"]) == set(model_names), "Drive model backup manifest has a different model set"
    for name in model_names:
        local_path = ARTIFACTS_ROOT / "models" / name
        backup_path = MODEL_BACKUP_ROOT / name
        assert local_path.is_file(), f"Missing trained model: {local_path}"
        expected_hash = manifest["models"][name]
        assert sha256_of_file(local_path) == expected_hash, f"Trained model differs from existing Drive backup: {name}"
        assert sha256_of_file(backup_path) == expected_hash, f"Drive model backup changed: {name}"
else:
    assert not MODEL_BACKUP_ROOT.exists() or not any(MODEL_BACKUP_ROOT.iterdir()), "Unverified partial Drive model backup exists; inspect before continuing"
    MODEL_BACKUP_ROOT.mkdir(parents=True, exist_ok=True)
    model_hashes = {}
    for name in model_names:
        local_path = ARTIFACTS_ROOT / "models" / name
        backup_path = MODEL_BACKUP_ROOT / name
        assert local_path.is_file(), f"Missing trained model: {local_path}"
        shutil.copy2(local_path, backup_path)
        digest = sha256_of_file(local_path)
        assert sha256_of_file(backup_path) == digest, f"Model backup verification failed: {name}"
        model_hashes[name] = digest
    MODEL_BACKUP_MANIFEST.write_text(json.dumps({"models": model_hashes}, indent=2, sort_keys=True) + "\n")
    print("Backed up six Phase 6 model files to Drive:", MODEL_BACKUP_ROOT)

In [ ]:
import json
from IPython.display import Markdown, display

result = json.loads((ARTIFACTS_ROOT / "training_results.json").read_text())
assert len(result["models"]) == 6
assert result["test_features_read"] is False
assert not (ARTIFACTS_ROOT / "FREEZE.lock").exists()
assert (ARTIFACTS_ROOT / "feature_choice.json").is_file()
assert (ARTIFACTS_ROOT / "primary_model.json").is_file()
display(Markdown((PROJECT_DIR / cfg["paths"]["reports_root"] / "phase_6_results.md").read_text()))
print("Selected frequency settings:", result["feature_choice"]["r0"], result["feature_choice"]["k_mad"])
print("Primary model:", result["primary_model"]["artifact_key"])
print("CV folds / searches per model:", result["cv_folds"], result["search_iterations_per_model"])


## STOP - Phase 6 team review
Review train/validation row counts, feature-choice scores, all six CV and validation metrics, saved versions/models, and descriptive importance plots. Confirm the split groups remain separated and check GitHub Actions. Agree and record the practical-use target in `reports/practical_target.md` before Phase 7. Keep real-data outputs and supervisor decisions pending until supplied; do not open `test_sealed` or create `FREEZE.lock` in this phase.

## Phase 7 - Freeze and single test evaluation
Run only after the team has reviewed the real Phase 6 training outputs and committed them with the agreed practical-use decision. This section first creates a hash lock without parsing the sealed test Parquet rows. Commit and push that lock to `main` before running the separate one-time evaluator.

The evaluator creates `artifacts/EVALUATION_STARTED.json` before opening test rows. If it fails after creating that marker, do not rerun it; preserve the marker and ask the supervisor how to proceed. No test data or research result is included in this notebook by default.

In [ ]:
import json
import shutil
from aidetect.freeze import validate_practical_target
from aidetect.io_utils import sha256_of_file

cfg = load_config(PROJECT_DIR / "config.yaml")
ARTIFACTS_ROOT = Path(cfg["paths"]["artifacts_root"])
REPORTS_ROOT = Path(cfg["paths"]["reports_root"])
branch = subprocess.check_output(["git", "branch", "--show-current"], cwd=PROJECT_DIR, text=True).strip()
status = subprocess.check_output(["git", "status", "--porcelain", "--untracked-files=all"], cwd=PROJECT_DIR, text=True).strip()
assert branch == "main", f"Phase 7 must run on main, not {branch}"
assert not status, "Commit and push reviewed Phase 6 outputs and the practical target before freezing"
assert not (ARTIFACTS_ROOT / "FREEZE.lock").exists(), "A freeze lock already exists"
assert not (ARTIFACTS_ROOT / "EVALUATION_STARTED.json").exists(), "The one-time evaluation has already started"
for name in ("feature_choice.json", "primary_model.json", "versions.json", "training_results.json"):
    assert (ARTIFACTS_ROOT / name).is_file(), f"Missing Phase 6 output: {name}"
model_names = [f"{feature_set}_{model}.joblib" for feature_set in ("frequency", "pixel") for model in ("svm", "rf", "xgb")]
model_files = {name: ARTIFACTS_ROOT / "models" / name for name in model_names}
# Preserve ignored model binaries across Colab runtime restarts. The freeze lock hashes the local files.
DRIVE_ROOT = Path("/content/drive/MyDrive/aidetect")
MODEL_BACKUP_ROOT = DRIVE_ROOT / "phase_6_models"
MODEL_BACKUP_MANIFEST = DRIVE_ROOT / "phase_6_models.sha256.json"
if MODEL_BACKUP_MANIFEST.is_file():
    manifest = json.loads(MODEL_BACKUP_MANIFEST.read_text())
    assert set(manifest["models"]) == set(model_names), "Drive model backup manifest has a different model set"
    for name, local_path in model_files.items():
        backup_path = MODEL_BACKUP_ROOT / name
        expected_hash = manifest["models"][name]
        if not local_path.exists():
            assert backup_path.is_file(), f"Missing backed-up model: {backup_path}"
            shutil.copy2(backup_path, local_path)
        assert sha256_of_file(backup_path) == expected_hash, f"Drive model backup changed: {name}"
        assert sha256_of_file(local_path) == expected_hash, f"Local model differs from reviewed backup: {name}"
else:
    assert not MODEL_BACKUP_ROOT.exists() or not any(MODEL_BACKUP_ROOT.iterdir()), "Unverified partial Drive model backup exists; inspect it before continuing"
    MODEL_BACKUP_ROOT.mkdir(parents=True, exist_ok=True)
    model_hashes = {}
    for name, local_path in model_files.items():
        assert local_path.is_file(), f"Missing Phase 6 model: {local_path}"
        backup_path = MODEL_BACKUP_ROOT / name
        shutil.copy2(local_path, backup_path)
        digest = sha256_of_file(local_path)
        assert sha256_of_file(backup_path) == digest, f"Model backup verification failed: {name}"
        model_hashes[name] = digest
    MODEL_BACKUP_MANIFEST.write_text(json.dumps({"models": model_hashes}, indent=2, sort_keys=True) + "\n")
    print("Backed up six Phase 6 model files to Drive:", MODEL_BACKUP_ROOT)

target = validate_practical_target(REPORTS_ROOT / "practical_target.md")
print("Ready on main commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=PROJECT_DIR, text=True).strip())
print("Agreed minimum validation accuracy:", target["minimum_validation_accuracy"])
print("Decision date:", target["decision_date"])
print("This preflight did not open test feature rows or values.")

In [ ]:
subprocess.run([sys.executable, "scripts/freeze.py"], cwd=PROJECT_DIR, check=True)

### STOP before test evaluation
Inspect `artifacts/FREEZE.lock`, then commit and push it to `main`. The evaluator refuses an uncommitted lock or dirty checkout. Do not run the next cell until the lock commit is on `main` and the checkout is clean.

```text
git add artifacts/FREEZE.lock
git commit -m "Freeze Phase 7 evaluation inputs"
git push origin main
git status --short --branch
```

In [ ]:
# A fresh Colab runtime may need to restore ignored model binaries from Drive. Never restore or rerun after an evaluation marker exists.
import json
import shutil
from aidetect.io_utils import sha256_of_file
cfg = load_config(PROJECT_DIR / "config.yaml")
ARTIFACTS_ROOT = Path(cfg["paths"]["artifacts_root"])
REPORTS_ROOT = Path(cfg["paths"]["reports_root"])
assert not (ARTIFACTS_ROOT / "EVALUATION_STARTED.json").exists(), "Evaluation already started; do not rerun"
assert not any((REPORTS_ROOT / name).exists() for name in ("results.json", "results.md", "test_predictions.csv")), "Evaluation outputs already exist; do not rerun"
DRIVE_ROOT = Path("/content/drive/MyDrive/aidetect")
MODEL_BACKUP_ROOT = DRIVE_ROOT / "phase_6_models"
manifest = json.loads((DRIVE_ROOT / "phase_6_models.sha256.json").read_text())
model_names = [f"{feature_set}_{model}.joblib" for feature_set in ("frequency", "pixel") for model in ("svm", "rf", "xgb")]
assert set(manifest["models"]) == set(model_names), "Drive model backup manifest has a different model set"
for name in model_names:
    local_path = ARTIFACTS_ROOT / "models" / name
    backup_path = MODEL_BACKUP_ROOT / name
    if not local_path.exists():
        shutil.copy2(backup_path, local_path)
    expected_hash = manifest["models"][name]
    assert sha256_of_file(backup_path) == expected_hash, f"Drive model backup changed: {name}"
    assert sha256_of_file(local_path) == expected_hash, f"Local model differs from reviewed backup: {name}"
subprocess.run([sys.executable, "scripts/evaluate_test.py"], cwd=PROJECT_DIR, check=True)

## STOP - Phase 7 team review
Review `reports/results.md`, `reports/results.json`, `reports/test_predictions.csv`, and `artifacts/EVALUATION_STARTED.json`. Hand-check at least one accuracy using `(TP + TN) / n` from its confusion matrix. Confirm all six models appear and H1, H1a, and H1b include their preregistered decisions, including non-rejections. Compare the primary test accuracy with the practical target recorded before freeze.

After a successful single run, commit and push the marker and result files to `main`. Do not rerun the evaluator. If it failed after the start marker appeared, do not rerun; retain the marker and contact the supervisor. Any later analysis must be labeled exploratory. Stop here until the team has reviewed the actual outputs and explicitly approves Phase 8.

## Phase 8 - Run the React prototype locally
The user authorized Phase 8 code while real-data review remains pending. See app/README.md and reports/phase_8_report.md. On the laptop: install Python requirements; run npm ci and npm run build in app/frontend; start python -m uvicorn app.backend.main:app --host 127.0.0.1 --port 8000 from the repository root. Open http://127.0.0.1:8000.

Use the reviewed Phase 7 model bundle/config/source for real predictions. Missing artifacts show an honest unavailable-model state. Check five reviewed images for probability parity, a text file renamed .jpg for a clear error, and note out-of-scope phone/screenshot behavior. Uploads remain in memory. Verify keyboard and physical mobile behavior. STOP after Phase 8 review; Phase 9 uses saved outputs only.

## Phase 9 - Report pack and reproducibility
TODO: add this phase's commands only when authorized. Read AGENTS.md and the PRD first.